In [31]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [32]:
df = spark.read.parquet(
    "s3a://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
)

In [33]:
important_cols=[
'FlightDate','Year','Quarter','Month','DayOfMonth','DayOfWeek',
'Operating_Airline','Marketing_Airline_Network',
'Flight_Number_Operating_Airline','Tail_Number',
'Origin','OriginCityName','OriginStateName',
'Dest','DestCityName','DestStateName',
'CRSDepTime','DepTime','CRSArrTime','ArrTime',
'DepDelay','DepDel15','ArrDelay','ArrDel15',
'Cancelled','CancellationCode','Diverted',
'TaxiOut','TaxiIn','AirTime','Distance'
]
available=[c for c in important_cols if c in df.columns]
eda_df=df.select(*available)
print("Columns:",len(available))
print(available)


Columns: 30
['FlightDate', 'Year', 'Quarter', 'Month', 'DayOfWeek', 'Operating_Airline', 'Marketing_Airline_Network', 'Flight_Number_Operating_Airline', 'Tail_Number', 'Origin', 'OriginCityName', 'OriginStateName', 'Dest', 'DestCityName', 'DestStateName', 'CRSDepTime', 'DepTime', 'CRSArrTime', 'ArrTime', 'DepDelay', 'DepDel15', 'ArrDelay', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'TaxiOut', 'TaxiIn', 'AirTime', 'Distance']

In [34]:
print(f"Total Rows    : {eda_df.count()}")
print(f"Total Columns : {len(eda_df.columns)}")

Total Rows    : 40910253
Total Columns : 30

In [35]:
from pyspark.sql.functions import col

dtype_df = spark.createDataFrame(
    [(c, t) for c, t in eda_df.dtypes],
    ["Column", "Data Type"]
)

dtype_df.show(truncate=False)

+-------------------------------+---------+
|Column                         |Data Type|
+-------------------------------+---------+
|FlightDate                     |timestamp|
|Year                           |int      |
|Quarter                        |int      |
|Month                          |int      |
|DayOfWeek                      |int      |
|Operating_Airline              |string   |
|Marketing_Airline_Network      |string   |
|Flight_Number_Operating_Airline|int      |
|Tail_Number                    |string   |
|Origin                         |string   |
|OriginCityName                 |string   |
|OriginStateName                |string   |
|Dest                           |string   |
|DestCityName                   |string   |
|DestStateName                  |string   |
|CRSDepTime                     |int      |
|DepTime                        |int      |
|CRSArrTime                     |int      |
|ArrTime                        |int      |
|DepDelay                       

In [36]:
eda_df.printSchema()

root
 |-- FlightDate: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Flight_Number_Operating_Airline: integer (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestStateName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDel15: double (nullable = true

In [37]:
from pyspark.sql.functions import count, when

null_df = eda_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
])

null_df.show(truncate=False)

+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------+------+-------+--------+
|FlightDate|Year|Quarter|Month|DayOfWeek|Operating_Airline|Marketing_Airline_Network|Flight_Number_Operating_Airline|Tail_Number|Origin|OriginCityName|OriginStateName|Dest|DestCityName|DestStateName|CRSDepTime|DepTime|CRSArrTime|ArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|Cancelled|CancellationCode|Diverted|TaxiOut|TaxiIn|AirTime|Distance|
+----------+----+-------+-----+---------+-----------------+-------------------------+-------------------------------+-----------+------+--------------+---------------+----+------------+-------------+----------+-------+----------+-------+--------+--------+--------+--------+---------+----------------+--------+-------

In [13]:
from pyspark.sql.functions import countDistinct

eda_df.agg(
    countDistinct("Marketing_Airline_Network").alias("Airlines"),
    countDistinct("Origin").alias("Origin Airports"),
    countDistinct("Dest").alias("Destination Airports"),
    countDistinct("OriginStateName").alias("Origin States"),
    countDistinct("DestStateName").alias("Destination States"),
    countDistinct("Year").alias("Years")
).show()

+--------+---------------+--------------------+-------------+------------------+-----+
|Airlines|Origin Airports|Destination Airports|Origin States|Destination States|Years|
+--------+---------------+--------------------+-------------+------------------+-----+
|      10|            390|                 391|           53|                53|    6|
+--------+---------------+--------------------+-------------+------------------+-----+

In [14]:
year_summary = eda_df.groupBy("Year").count().orderBy("Year")
year_summary.show()

+----+-------+
|Year|  count|
+----+-------+
|2020|5022397|
|2021|6311871|
|2022|7013508|
|2023|7278739|
|2024|7546968|
|2025|7736770|
+----+-------+

In [17]:
from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    ShortType,
    DecimalType
)

In [18]:
numeric = [
    f.name
    for f in eda_df.schema.fields
    if isinstance(
        f.dataType,
        (IntegerType, LongType, DoubleType, FloatType, ShortType, DecimalType)
    )
]

print(numeric)

eda_df.select(numeric).describe().show()

['Year', 'Quarter', 'Month', 'DayOfWeek', 'Flight_Number_Operating_Airline', 'CRSDepTime', 'DepTime', 'CRSArrTime', 'ArrTime', 'DepDelay', 'DepDel15', 'ArrDelay', 'ArrDel15', 'Cancelled', 'Diverted', 'TaxiOut', 'TaxiIn', 'AirTime', 'Distance']
+-------+------------------+------------------+------------------+------------------+-------------------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+-----------------+------------------+-------------------+--------------------+-----------------+-----------------+------------------+-----------------+
|summary|              Year|           Quarter|             Month|         DayOfWeek|Flight_Number_Operating_Airline|        CRSDepTime|          DepTime|        CRSArrTime|          ArrTime|          DepDelay|          DepDel15|         ArrDelay|          ArrDel15|          Cancelled|            Diverted|          TaxiOut|           TaxiIn|           AirTime|         Dist

In [22]:
from pyspark.sql.functions import desc

categorical_columns = [
    "Operating_Airline",
    "Marketing_Airline_Network",
    "Origin",
    "OriginCityName",
    "OriginStateName",
    "Dest",
    "DestCityName",
    "DestStateName",
    "CancellationCode"
]

for c in categorical_columns:
    print("\n" + "="*80)
    print(f"Top 10 values for: {c}")
    print("="*80)

    eda_df.groupBy(c) \
          .count() \
          .orderBy(desc("count")) \
          .show(10, truncate=False)


Top 10 values for: Operating_Airline
+-----------------+-------+
|Operating_Airline|count  |
+-----------------+-------+
|WN               |7582834|
|DL               |5242763|
|AA               |5078840|
|OO               |4343633|
|UA               |3669461|
|YX               |1812974|
|MQ               |1521850|
|B6               |1366470|
|9E               |1354202|
|OH               |1300460|
+-----------------+-------+
only showing top 10 rows


Top 10 values for: Marketing_Airline_Network
+-------------------------+--------+
|Marketing_Airline_Network|count   |
+-------------------------+--------+
|AA                       |10407897|
|DL                       |8526129 |
|WN                       |7582834 |
|UA                       |7427635 |
|AS                       |2238407 |
|B6                       |1366470 |
|NK                       |1278352 |
|F9                       |968030  |
|G4                       |694895  |
|HA                       |419604  |
+----------------

In [23]:
from pyspark.sql.functions import desc

eda_df.groupBy("Operating_Airline") \
      .count() \
      .orderBy(desc("count")) \
      .show(10, False)

+-----------------+-------+
|Operating_Airline|count  |
+-----------------+-------+
|WN               |7582834|
|DL               |5242763|
|AA               |5078840|
|OO               |4343633|
|UA               |3669461|
|YX               |1812974|
|MQ               |1521850|
|B6               |1366470|
|9E               |1354202|
|OH               |1300460|
+-----------------+-------+
only showing top 10 rows

In [24]:
eda_df.groupBy("Origin") \
      .count() \
      .orderBy(desc("count")) \
      .show(10, False)

+------+-------+
|Origin|count  |
+------+-------+
|ATL   |1915120|
|ORD   |1778804|
|DFW   |1705528|
|DEN   |1683092|
|CLT   |1347556|
|LAX   |1086362|
|SEA   |1026212|
|PHX   |1021572|
|LAS   |999915 |
|IAH   |904228 |
+------+-------+
only showing top 10 rows

In [27]:

eda_df.agg(
    avg("DepDelay").alias("Average Departure Delay"),
    avg("ArrDelay").alias("Average Arrival Delay"),
    max("DepDelay").alias("Maximum Departure Delay"),
    max("ArrDelay").alias("Maximum Arrival Delay")
).show()

+-----------------------+---------------------+-----------------------+---------------------+
|Average Departure Delay|Average Arrival Delay|Maximum Departure Delay|Maximum Arrival Delay|
+-----------------------+---------------------+-----------------------+---------------------+
|     10.937542346396805|    5.242046383623324|                 7223.0|               7232.0|
+-----------------------+---------------------+-----------------------+---------------------+

In [28]:
print("Distance vs AirTime")

print(eda_df.stat.corr("Distance","AirTime"))

print("Departure Delay vs Arrival Delay")

print(eda_df.stat.corr("DepDelay","ArrDelay"))

Distance vs AirTime
0.9467148729106268
Departure Delay vs Arrival Delay
0.9626657385544707

In [26]:
from pyspark.sql.functions import (
    avg,
    min,
    max,
    sum,
    count,
    desc,
    col,
    when,
    countDistinct,
    concat_ws
)

In [29]:
print("Distance vs AirTime")

print(eda_df.stat.corr("Distance","AirTime"))

print("Departure Delay vs Arrival Delay")

print(eda_df.stat.corr("DepDelay","ArrDelay"))

Distance vs AirTime
0.9467148729106268
Departure Delay vs Arrival Delay
0.9626657385544707

In [30]:
eda_df.groupBy("ArrDel15").count().show()
eda_df.groupBy("DepDel15").count().show()
eda_df.groupBy("Cancelled").count().show()

+--------+--------+
|ArrDel15|   count|
+--------+--------+
|     0.0|32251102|
|    null| 1014879|
|     1.0| 7644272|
+--------+--------+

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32448294|
|    null|  896518|
|     1.0| 7565441|
+--------+--------+

+---------+--------+
|Cancelled|   count|
+---------+--------+
|      0.0|39993169|
|      1.0|  917084|
+---------+--------+

In [38]:
dep_violations = eda_df.filter(
    ((col("DepDelay") >= 15) & (col("DepDel15") != 1)) |
    ((col("DepDelay") < 15) & (col("DepDel15") != 0) & col("DepDelay").isNotNull())
).count()
print(f"DepDel15 violations: {dep_violations:,}")

DepDel15 violations: 0

In [39]:
eda_df.filter(col("ArrDelay").isNull()) \
    .groupBy("Cancelled", "Diverted") \
    .count() \
    .orderBy(desc("count")) \
    .show()

+---------+--------+------+
|Cancelled|Diverted| count|
+---------+--------+------+
|      1.0|     0.0|917084|
|      0.0|     1.0| 97790|
|      0.0|     0.0|     5|
+---------+--------+------+

In [40]:
key_cols = ["FlightDate","Marketing_Airline_Network","Flight_Number_Operating_Airline","Origin","Dest","CRSDepTime"]
total_rows = eda_df.count()
distinct_count = eda_df.select(key_cols).distinct().count()
print(f"Total: {total_rows:,} | Distinct: {distinct_count:,} | Unique?", distinct_count == total_rows)

Total: 40,910,253 | Distinct: 40,910,253 | Unique? True

In [41]:
eda_df.groupBy("Operating_Airline") \
    .agg(count("*").alias("flights"),
         round(avg("DepDelay"),2).alias("avg_dep_delay"),
         round(avg("Cancelled"),4).alias("cancel_rate")) \
    .orderBy(desc("avg_dep_delay")) \
    .show(20, False)

+-----------------+-------+-------------+-----------+
|Operating_Airline|flights|avg_dep_delay|cancel_rate|
+-----------------+-------+-------------+-----------+
|B6               |1366470|18.1         |0.0255     |
|F9               |968030 |17.31        |0.024      |
|AA               |5078840|15.57        |0.0235     |
|G4               |694895 |14.45        |0.0369     |
|NK               |1278352|13.78        |0.0219     |
|YV               |670438 |13.66        |0.0322     |
|G7               |327667 |13.23        |0.0379     |
|C5               |413126 |12.12        |0.0245     |
|ZW               |322146 |11.65        |0.0307     |
|WN               |7582834|11.35        |0.024      |
|UA               |3669461|11.09        |0.019      |
|OH               |1300460|10.88        |0.0311     |
|OO               |4343633|9.82         |0.0184     |
|DL               |5242763|8.94         |0.0147     |
|AX               |18705  |7.65         |0.0865     |
|MQ               |1521850|7